# All source code and data explorer

**Author: Mohammad ALbaraj**

[Notebook index](README.md) · [Main project](../README.md)

Run cells from top to bottom. Saved results and fresh examples are labeled separately. Scientific source modules remain the authoritative implementation.


In [1]:
from pathlib import Path
import sys, json, csv
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Code

# Works when launched from the repository root or any notebook subdirectory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'CITATION.cff').exists() and (p / 'all work').is_dir())
if str(ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(ROOT / 'notebooks'))
from notebook_support import (METHODS, ORIGINAL, NOMINAL, ROBUST, study, load_json,
    load_csv, show_table, show_source, source_index, load_runs, fitness_summary,
    convergence_plot, validation_rows, validation_summary, figures, file_links,
    show_figures, gallery, isolated_python, demo_source)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})
print('Project loaded. Default cells do not launch full optimization campaigns.')


Project loaded. Default cells do not launch full optimization campaigns.


## Complete Python source index
This includes the original 278 Python source files, their functions/classes, and the relative paths. Scientific code stays in its source files to avoid notebook copies diverging from frozen experiment hashes.


In [2]:
index=source_index()
print("Source modules:",len(index))
show_table(index,limit=len(index))


Source modules: 278


file,definitions
all work/30 seed equal budget study/analyze.py,"uncertainty_cases, main"
all work/30 seed equal budget study/benchmark.py,"BudgetExhausted, Budget, initial_population, optimize, main"
all work/30 seed equal budget study/build_pdf.py,"rows, text, start, note, table, picture, bullets"
all work/30 seed equal budget study/dsa_reference.py,"predation_radius, communicate, dolphin_swarm"
all work/30 seed equal budget study/inspect_source.py,inventory
all work/30 seed equal budget study/linear_analysis.py,local_analysis
all work/30 seed equal budget study/model.py,"baseline, coefficients, rc_condition, euler, rhs, simulate, run, rotation_matrix, full_rhs"
all work/30 seed equal budget study/prepare_delivery.py,main
all work/30 seed equal budget study/run_parallel.py,"worker, main"
all work/30 seed equal budget study/study.py,"training_scenarios, validation_scenarios, robust_training_scenarios, assess, evaluate, decode, encode, write_csv, metric_rows"


## Inspect any source module
Change SOURCE_FILE and SYMBOL. Leave SYMBOL as None to display the complete file.


In [3]:
SOURCE_FILE='pid using alns/methods.py'
SYMBOL=None
show_source(ROOT/SOURCE_FILE,SYMBOL)


"""Six explicitly specified bounded continuous-search variants.
All operate in normalized log-gain coordinates; the original objective is unchanged.
"""
from pathlib import Path
import json,time,hashlib
import numpy as np
from benchmark import Budget,BudgetExhausted,initial_population
from study import decode
METHODS=['GDA','TA','ALNS','VND','SBRS','PRW']
FULL_NAMES={'GDA':'Great Deluge Algorithm','TA':'Threshold Accepting','ALNS':'Adaptive Large Neighborhood Search','VND':'Variable Neighborhood Descent (budgeted multistart)','SBRS':'Shrinking Ball Random Search','PRW':'Pure Random Walk (constant proposal step)'}
SEEDS=[11,29,47]+list(range(101,128))
PILOT_SEEDS=[5011,5021,5039]

def reflect(x):
    y=np.mod(x,2.);return np.where(y<=1.,y,2.-y)

def direction(rng,d):
    while True:
        z=rng.normal(size=d);norm=np.linalg.norm(z)
        if norm>0:return z/norm

def merit(f):return float(np.log1p(f))

def accept(method,current,trial,level):
    if method=='PRW':return True
    if method=='GDA':return trial<=current or merit(trial)<=level
    if method in ['TA','ALNS']:return merit(trial)<=merit(current)+level
    return trial<current

def parameter_grid(method):
    if method=='GDA':return [{'step':s,'water_slack':a} for s in [.03,.12] for a in [.05,.25]]
    if method=='TA':return [{'step':s,'threshold0':a} for s in [.03,.12] for a in [.02,.20]]
    if method=='ALNS':return [{'repair_sigma':s,'reaction':r,'threshold0':.05,'segment':64} for s in [.04,.16] for r in [.1,.4]]
    if method=='VND':return [{'radius':r,'neighborhood_order':o} for r in [.05,.20] for o in ['large_to_small','small_to_large']]
    if method=='SBRS':return [{'radius0':r,'radius_ratio':q} for r in [.15,.50] for q in [.01,.1]]
    if method=='PRW':return [{'step':r} for r in [.01,.04,.12,.30]]
    raise ValueError(method)

def optimize(method,seed,cfg,params,out=None,limit=3264,objective=None):
    if method not in METHODS:raise ValueError(method)
    if limit<65:raise ValueError('At least 65 evaluations required for shared initialization plus search')
    rng,x0=initial_population(seed,cfg);init_hash=hashlib.sha256(x0.tobytes()).hexdigest()
    b=Budget(cfg,limit,None,objective);events=[];restarts=0;accepted=0
    try:
        f0=b.score(x0);idx=int(np.argmin(f0));x=x0[idx].copy();f=float(f0[idx]);initial_best=f
        water0=merit(f)+params.get('water_slack',0.)
        wd=np.ones(4);wr=np.ones(3);sd=np.zeros(4);sr=np.zeros(3);nd=np.zeros(4);nr=np.zeros(3)
        radii=params.get('radius',.1)*np.array([1.,.5,.1,.02])
        if params.get('neighborhood_order')=='small_to_large':radii=radii[::-1]
        k=0;coord=0;cycle_start=x.copy()
        while True:
            b.iteration+=1;p=b.progress();level=0.;opd=-1;opr=-1;restart=False
            if method in ['GDA','TA','PRW']:
                trial=reflect(x+params['step']*direction(rng,len(x)))
                level=water0*(1-p) if method=='GDA' else params.get('threshold0',0.)*(1-p)
            elif method=='SBRS':
                radius=params['radius0']*params['radius_ratio']**p
                trial=reflect(x+radius*rng.random()**(1/len(x))*direction(rng,len(x)))
                level=radius
            elif method=='VND':
                if k==len(radii):
                    trial=rng.random(len(x));restart=True;restarts+=1;level=-1.
                else:
                    trial=x.copy();trial[coord//2]+=radii[k]*(1 if coord%2==0 else -1);trial=reflect(trial);level=float(radii[k])
            else:
                opd=int(rng.choice(4,p=wd/wd.sum()));opr=int(rng.choice(3,p=wr/wr.sum()))
                dims=rng.choice(len(x),[1,2,4,8][opd],replace=False);trial=x.copy()
                if opr==0:trial[dims]=rng.random(len(dims))
                elif opr==1:trial[dims]+=rng.normal(0,params['repair_sigma'],len(dims))
                else:
                    elite=np.array([row[4:] for row in sorted(b.ledger,key=lambda row:row[2])[:16]])
                    a,c,d=elite[rng.choice(len(elite),3,r

## MATLAB and Simulink files
The notebooks show initialization code and model-file locations. Running an SLX model requires MATLAB/Simulink outside this Python collection.


In [4]:
file_links(sorted(p for p in ROOT.rglob('*') if p.suffix in {'.m','.slx'}),'Model source files')
show_source(study('GA')/'source/Init_control.m')


### Model source files
- [all work/30 seed equal budget study/source/AttitudeControl_Sim.slx](../all%20work/30%20seed%20equal%20budget%20study/source/AttitudeControl_Sim.slx)
- [all work/30 seed equal budget study/source/Init.m](../all%20work/30%20seed%20equal%20budget%20study/source/Init.m)
- [all work/30 seed equal budget study/source/Init_control.m](../all%20work/30%20seed%20equal%20budget%20study/source/Init_control.m)
- [all work/pid using cat swarm/source/AttitudeControl_Sim.slx](../all%20work/pid%20using%20cat%20swarm/source/AttitudeControl_Sim.slx)
- [all work/pid using cat swarm/source/Init.m](../all%20work/pid%20using%20cat%20swarm/source/Init.m)
- [all work/pid using cat swarm/source/Init_control.m](../all%20work/pid%20using%20cat%20swarm/source/Init_control.m)
- [all work/pid using cma es/source/AttitudeControl_Sim.slx](../all%20work/pid%20using%20cma%20es/source/AttitudeControl_Sim.slx)
- [all work/pid using cma es/source/Init.m](../all%20work/pid%20using%20cma%20es/source/Init.m)
- [all work/pid using cma es/source/Init_control.m](../all%20work/pid%20using%20cma%20es/source/Init_control.m)
- [all work/pid using dolphin swarm/source/AttitudeControl_Sim.slx](../all%20work/pid%20using%20dolphin%20swarm/source/AttitudeControl_Sim.slx)
- [all work/pid using dolphin swarm/source/Init.m](../all%20work/pid%20using%20dolphin%20swarm/source/Init.m)
- [all work/pid using dolphin swarm/source/Init_control.m](../all%20work/pid%20using%20dolphin%20swarm/source/Init_control.m)
- [all work/pid using ga/source/AttitudeControl_Sim.slx](../all%20work/pid%20using%20ga/source/AttitudeControl_Sim.slx)
- [all work/pid using ga/source/Init.m](../all%20work/pid%20using%20ga/source/Init.m)
- [all work/pid using ga/source/Init_control.m](../all%20work/pid%20using%20ga/source/Init_control.m)
- [all work/pid using pso/source/AttitudeControl_Sim.slx](../all%20work/pid%20using%20pso/source/AttitudeControl_Sim.slx)
- [all work/pid using pso/source/Init.m](../all%20work/pid%20using%20pso/source/Init.m)
- [all work/pid using pso/source/Init_control.m](../all%20work/pid%20using%20pso/source/Init_control.m)
- [all work/robustness 30 seed study/source/AttitudeControl_Sim.slx](../all%20work/robustness%2030%20seed%20study/source/AttitudeControl_Sim.slx)
- [all work/robustness 30 seed study/source/Init.m](../all%20work/robustness%2030%20seed%20study/source/Init.m)
- [all work/robustness 30 seed study/source/Init_control.m](../all%20work/robustness%2030%20seed%20study/source/Init_control.m)
- [pid using alns/source/AttitudeControl_Sim.slx](../pid%20using%20alns/source/AttitudeControl_Sim.slx)
- [pid using alns/source/Init.m](../pid%20using%20alns/source/Init.m)
- [pid using alns/source/Init_control.m](../pid%20using%20alns/source/Init_control.m)
- [pid using great deluge/source/AttitudeControl_Sim.slx](../pid%20using%20great%20deluge/source/AttitudeControl_Sim.slx)
- [pid using great deluge/source/Init.m](../pid%20using%20great%20deluge/source/Init.m)
- [pid using great deluge/source/Init_control.m](../pid%20using%20great%20deluge/source/Init_control.m)
- [pid using pure random walk/source/AttitudeControl_Sim.slx](../pid%20using%20pure%20random%20walk/source/AttitudeControl_Sim.slx)
- [pid using pure random walk/source/Init.m](../pid%20using%20pure%20random%20walk/source/Init.m)
- [pid using pure random walk/source/Init_control.m](../pid%20using%20pure%20random%20walk/source/Init_control.m)
- [pid using shrinking ball random search/source/AttitudeControl_Sim.slx](../pid%20using%20shrinking%20ball%20random%20search/source/AttitudeControl_Sim.slx)
- [pid using shrinking ball random search/source/Init.m](../pid%20using%20shrinking%20ball%20random%20search/source/Init.m)
- [pid using shrinking ball random search/source/Init_control.m](../pid%20using%20shrinking%20ball%20random%20search/source/Init_control.m)
- [pid using threshold accepting/source/AttitudeControl_Sim.slx](../pid%20using%20threshold%20accepting/source/AttitudeControl_Sim.slx)
- [pid using threshold accepting/source/Init.m](../pid%20using%20threshold%20accepting/source/Init.m)
- [pid using threshold accepting/source/Init_control.m](../pid%20using%20threshold%20accepting/source/Init_control.m)
- [pid using vnd/source/AttitudeControl_Sim.slx](../pid%20using%20vnd/source/AttitudeControl_Sim.slx)
- [pid using vnd/source/Init.m](../pid%20using%20vnd/source/Init.m)
- [pid using vnd/source/Init_control.m](../pid%20using%20vnd/source/Init_control.m)

%%%  AttitudeControl_Sim
clear
path('./icon/',path);
%Run the model initialization file icon/init.m
Init;

%Constant value
RAD2DEG = 57.2957795;
DEG2RAD = 0.0174533;
%throttle when UAV is hovering
THR_HOVER = 0.609;

%% Initial condition
ModelInit_PosE = [0, 0, 0];
ModelInit_VelB = [0, 0, 0];
ModelInit_AngEuler = [0, 0, 0];
ModelInit_RateB = [0, 0, 0];
ModelInit_Rads = 0;
%% control parameter
%Attitude PID parameters
Kp_YAW_AngleRate = 0.5;
Ki_YAW_AngleRate = 0.01;
Kd_YAW_AngleRate = 0.00;
%integral saturation
Saturation_I_RP_Max = 0.3;
Saturation_I_RP_Min = -0.3;
Saturation_I_Y_Max = 0.2;
Saturation_I_Y_Min = -0.2;
%max control angle,default 35deg
MAX_CONTROL_ANGLE_ROLL = 35;
MAX_CONTROL_ANGLE_PITCH  = 35;
%max control angle rate,rad/s 
MAX_CONTROL_ANGLE_RATE_PITCH = 220;
MAX_CONTROL_ANGLE_RATE_ROLL = 220;
MAX_CONTROL_ANGLE_RATE_Y = 200;
%% run simulink model
AttitudeControl_Sim

## Complete numerical data manifest
Each archived file has its size and SHA-256 digest. The default view groups by suffix; filtering lets you inspect every entry without printing thousands of rows.


In [5]:
from collections import Counter
manifest=load_csv(ROOT/'docs/results_manifest.csv')
counts=Counter(Path(r['repository_path']).suffix for r in manifest)
show_table([{'suffix':k,'files':v} for k,v in sorted(counts.items())])
FILTER='pid using alns/results/exports'
show_table([r for r in manifest if FILTER in r['repository_path']],limit=100)


suffix,files
.csv,481
.gz,742
.json,3665
.npz,1044
.png,166
.svg,64
.txt,54


repository_path,bytes,sha256
pid using alns/results/exports/all_runs_gains.csv,17564,089d30151f0a246c085fb7c6cd29e4a0afaaa6e955f0f6c678adbe817603722b
pid using alns/results/exports/nominal_ALNS_best_gains.json,2579,151d7c9cb35bf2393db931e69b9da58766d8faa433f93e7e3fa225bf8c62cef3
pid using alns/results/exports/nominal_ALNS_best_SIMULINK_MAP.txt,2579,151d7c9cb35bf2393db931e69b9da58766d8faa433f93e7e3fa225bf8c62cef3
pid using alns/results/exports/nominal_ALNS_median_gains.json,2574,16bc0f97e403361541c1b9e4cb4f65cd6148af028fcb66ff6e9cdf412155817f
pid using alns/results/exports/nominal_ALNS_median_SIMULINK_MAP.txt,2574,16bc0f97e403361541c1b9e4cb4f65cd6148af028fcb66ff6e9cdf412155817f
pid using alns/results/exports/robust_ALNS_best_gains.json,2563,8d1e327119cfde60ac9db412bdb52615f784df5f1e4ecbd8e004445c3fd50097
pid using alns/results/exports/robust_ALNS_best_SIMULINK_MAP.txt,2563,8d1e327119cfde60ac9db412bdb52615f784df5f1e4ecbd8e004445c3fd50097
pid using alns/results/exports/robust_ALNS_median_gains.json,2566,af433b85f12112af0c9635ec4784f51be63ac7ee6907a96f50c277b2911f8d5c
pid using alns/results/exports/robust_ALNS_median_SIMULINK_MAP.txt,2566,af433b85f12112af0c9635ec4784f51be63ac7ee6907a96f50c277b2911f8d5c


## Read CSV, JSON, NPZ, or compressed JSON
No pickle loading is needed. Change DATA_FILE; tables are truncated for readability, not modified.


In [6]:
DATA_FILE='pid using alns/results/nominal/runs/ALNS_11_evaluations.npz'
path=ROOT/DATA_FILE
if path.suffix=='.csv':
    show_table(load_csv(path),limit=20)
elif path.suffix=='.npz':
    with np.load(path,allow_pickle=False) as arrays:
        show_table([{'array':k,'shape':str(arrays[k].shape),'dtype':str(arrays[k].dtype)} for k in arrays.files])
        if 'ledger' in arrays.files: print(arrays['ledger'][:3])
else:
    print(json.dumps(load_json(path),indent=2)[:6000])


array,shape,dtype
ledger,"(3264, 12)",float64
transitions,"(3210, 9)",float64


[[1.00000000e+00 0.00000000e+00 1.01161676e+03 1.01161676e+03
  4.73896368e-01 4.38687046e-01 8.09561942e-01 4.73896368e-01
  4.38687046e-01 8.09561942e-01 8.49485002e-01 5.40691057e-01]
 [2.00000000e+00 0.00000000e+00 1.69394345e+03 1.01161676e+03
  9.03798354e-01 8.05317961e-01 7.51171568e-01 2.44957030e-01
  1.77034430e-01 7.72180113e-01 9.41322130e-01 7.75840650e-01]
 [3.00000000e+00 0.00000000e+00 2.21369643e+05 1.01161676e+03
  6.10084873e-01 7.58470686e-01 7.88263669e-01 9.51547745e-01
  4.56303807e-01 8.94338678e-01 8.79453796e-01 3.30713540e-01]]


## Notebook support source
The helper module handles rendering and read-only access. Scientific modules run in isolated processes to avoid collisions between identically named model.py/study.py files.


In [7]:
show_source(ROOT/"notebooks/notebook_support.py")


"""Read-only study exploration and isolated, opt-in experiment helpers."""
from pathlib import Path
import ast
import csv
import gzip
import html
import json
import os
import subprocess
import sys
from urllib.parse import quote

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Code, HTML, Image, Markdown, display

ROOT = Path(__file__).resolve().parents[1]
METHODS = {
    'GA': ('all work/pid using ga', 'Genetic Algorithm', 'study.py'),
    'PSO': ('all work/pid using pso', 'Particle Swarm Optimization', 'pso.py'),
    'CSO': ('all work/pid using cat swarm', 'Cat Swarm Optimization', 'cso.py'),
    'DSA': ('all work/pid using dolphin swarm', 'Dolphin Swarm Algorithm', 'dsa.py'),
    'CMA-ES': ('all work/pid using cma es', 'Covariance Matrix Adaptation Evolution Strategy', 'cmaes.py'),
    'GDA': ('pid using great deluge', 'Great Deluge Algorithm', 'methods.py'),
    'TA': ('pid using threshold accepting', 'Threshold Accepting', 'methods.py'),
    'ALNS': ('pid using alns', 'Adaptive Large Neighborhood Search', 'methods.py'),
    'VND': ('pid using vnd', 'Variable Neighborhood Descent', 'methods.py'),
    'SBRS': ('pid using shrinking ball random search', 'Shrinking Ball Random Search', 'methods.py'),
    'PRW': ('pid using pure random walk', 'Pure Random Walk', 'methods.py'),
}
ORIGINAL = list(METHODS)[:5]
NOMINAL = ROOT / 'all work/30 seed equal budget study'
ROBUST = ROOT / 'all work/robustness 30 seed study'


def study(method):
    return ROOT / METHODS[method][0]


def load_json(path):
    path = Path(path)
    if path.name.endswith('.json.gz'):
        with gzip.open(path, 'rt', encoding='utf-8') as stream:
            return json.load(stream)
    return json.loads(path.read_text(encoding='utf-8-sig'))


def load_csv(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))


def show_table(rows, columns=None, limit=40):
    rows = list(rows)
    if not rows:
        display(Markdown('*No matching rows.*'))
        return
    columns = columns or list(rows[0])
    def cell(v):
        if isinstance(v, (dict, list, tuple)):
            v = json.dumps(v, ensure_ascii=False)
        return html.escape(str(v))
    text = '<table><thead><tr>' + ''.join('<th>'+cell(c)+'</th>' for c in columns) + '</tr></thead><tbody>'
    for row in rows[:limit]:
        text += '<tr>' + ''.join('<td>'+cell(row.get(c, ''))+'</td>' for c in columns) + '</tr>'
    display(HTML(text+'</tbody></table>'))
    if len(rows) > limit:
        print(f'Showing {limit} of {len(rows)} rows. Increase limit to inspect more.')


def show_source(path, symbol=None):
    path = Path(path)
    source = path.read_text(encoding='utf-8-sig')
    if symbol:
        tree = ast.parse(source)
        node = next(n for n in tree.body if getattr(n, 'name', None) == symbol)
        source = '\n'.join(source.splitlines()[node.lineno-1:node.end_lineno])
    display(Code(source, language='python' if path.suffix == '.py' else 'text'))


def source_index():
    result = []
    for p in sorted(ROOT.rglob('*.py')):
        if any(part in {'.git', '.venv', '.venv-notebooks', 'notebook_outputs'} for part in p.parts) or p.is_relative_to(ROOT/'notebooks'):
            continue
        tree = ast.parse(p.read_text(encoding='utf-8-sig'))
        names = [n.name for n in tree.body if isinstance(n, (ast.FunctionDef, ast.ClassDef))]
        result.append({'file': p.relative_to(ROOT).as_posix(), 'definitions': ', '.join(names)})
    return result


def load_runs(method, mode='nominal'):
    if mode not in {'nominal', 'robust'}:
        raise ValueError('mode must be nominal or robust')
    folder = (NOMINAL if mode == 'nominal' else ROBUST)/'results/runs' if method in ORIGINAL else study(method)/'results'/mode/'runs'
    runs = [(p, load_json(p)) for p in sorted(folder.glob(f'{method}_*.json'))
            if p.stem.removeprefix(method + '_').isdigit()]
    assert len(runs) == 30, (method, mode, le